# 🏦 Investment Research with Compliance Plan Review

## Overview

This notebook demonstrates **interactive plan review** in Magentic workflows for financial services - enabling compliance oversight and modification of research plans before execution. This pattern is essential for:

1. **Regulatory Compliance**: Ensure research follows SEC/FINRA guidelines
2. **Risk Management**: Human approval for client-facing materials
3. **Quality Assurance**: Verify AI-generated research methodology
4. **Iterative Refinement**: Collaborate with AI to optimize research approach

> ⚠️ **Financial Services Disclaimer**: This notebook demonstrates AI agent workflows for educational purposes. In production, all investment research must be reviewed by compliance before distribution.

### Industry Use Case: Compliance-Reviewed Investment Research

A wealth management firm requires compliance review of AI-generated research plans before execution to ensure:
- Research methodology meets regulatory standards
- Appropriate disclosures are included
- Data sources are approved and documented
- Client suitability considerations are addressed

### Key Concepts

| Concept | Description |
|---------|-------------|
| **`.with_plan_review()`** | Enables human review of generated plans |
| **`MagenticPlanReviewRequest`** | Event type for plan review requests |
| **`event_data.approve()`** | Accept the proposed plan |
| **`event_data.revise(feedback)`** | Provide feedback to modify the plan |

### Plan Review Lifecycle

```
Investment Research Request
           ↓
Orchestrator Generates Research Plan
           ↓
Request Compliance Review (with_plan_review)
           ↓
┌─────────────────────────────────┐
│   Compliance Reviews Plan       │
│  ├── approve() - Accept plan    │
│  └── revise(feedback) - Modify  │
└─────────────────────────────────┘
           ↓
Resume with Compliance Decision
           ↓
Execute Plan → Final Research Report
```

### Review Response Options

| Method | Action | Industry Use Case |
|--------|--------|--------------|
| **`approve()`** | Accept plan, continue execution | Research plan meets compliance standards |
| **`revise(feedback)`** | Modify plan with feedback | Add disclosures or adjust methodology |

## Prerequisites

- ✅ Azure AI Foundry configured
- ✅ Environment variables: `AI_FOUNDRY_PROJECT_ENDPOINT`, `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- ✅ Azure CLI authentication: Run `az login`

## 1️⃣ Setup and Imports

In [ ]:
import asyncio
import json
import logging
from typing import cast

import os
from dotenv import load_dotenv
from agent_framework import (
    Agent,
    AgentResponse,
    AgentResponseUpdate,
    WorkflowEvent,
)
from agent_framework.orchestrations import (
    MagenticBuilder,
    MagenticPlanReviewRequest,
    MagenticPlanReviewResponse,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(__name__)

# Load environment variables from .env file
load_dotenv('../../.env')

## 2️⃣ Create Specialized Financial Agents

Multi-agent setup for investment research:
- **MarketResearcherAgent**: Market data, sector analysis, news gathering
- **QuantAnalystAgent**: Data analysis, summarization, insights
- **MagenticManager**: Orchestrator that coordinates the workflow

In [ ]:
# Create Foundry chat client
PROJECT_ENDPOINT = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
chat_client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL_DEPLOYMENT,
    credential=AzureCliCredential()
)
print("✅ Foundry Chat Client created")

# Create specialized financial agents
market_researcher = Agent(
    client=chat_client,
    name="MarketResearcherAgent",
    description="Specialist in market research, sector analysis, and financial news gathering",
    instructions=(
        "You are a Financial Market Researcher at a wealth management firm. "
        "You gather market data, sector trends, company news, and economic indicators. "
        "Focus on factual information without performing calculations. "
        "Always cite sources and note the date of any market data you reference."
    ),
)
print("✅ Market Researcher Agent created")

quant_analyst = Agent(
    client=chat_client,
    name="QuantAnalystAgent",
    description="Data analyst who processes and summarizes research findings",
    instructions=(
        "You are a Quantitative Analyst at a wealth management firm. "
        "You analyze findings and create summaries with risk metrics. "
        "Always show your calculation methodology and assumptions."
    ),
)
print("✅ Quant Analyst Agent created")

## 3️⃣ Build Magentic Workflow with Plan Review

Use `MagenticBuilder` with `enable_plan_review=True` to enable compliance review of generated plans before execution.

The Magentic manager dynamically coordinates agents and generates a plan that requires compliance approval via `MagenticPlanReviewRequest`/`MagenticPlanReviewResponse`.

In [ ]:
print("\n🏦 Building Investment Research Workflow with Compliance Plan Review...")

# Create a manager agent for orchestration
manager_agent = Agent(
    client=chat_client,
    name="ResearchManager",
    description="Orchestrator that coordinates the investment research workflow",
    instructions=(
        "You coordinate a team of financial analysts to complete investment research tasks efficiently. "
        "Review outputs from the Market Researcher and pass relevant findings to the Quant Analyst. "
        "Synthesize a final comprehensive investment research report. "
        "Ensure all research includes appropriate risk disclosures."
    ),
)
print("✅ Research Manager Agent created")

# Build the Magentic workflow with plan review enabled for compliance oversight
workflow = MagenticBuilder(
    participants=[market_researcher, quant_analyst],
    intermediate_output_from=[market_researcher, quant_analyst],
    manager_agent=manager_agent,
    enable_plan_review=True,  # Enables human-in-the-loop compliance review
    max_round_count=10,
    max_stall_count=3,
    max_reset_count=2,
).build()

print("✅ Magentic workflow built with compliance plan review enabled")
print("\nWorkflow Configuration:")
print("  • Participants: MarketResearcherAgent, QuantAnalystAgent")
print("  • Manager: ResearchManager (orchestrates dynamically)")
print("  • Plan Review: ENABLED (compliance must approve before execution)")
print("  • Max Rounds: 10 | Max Stalls: 3 | Max Resets: 2")

## 4️⃣ Define Investment Research Request

In [ ]:
# Client research request from a wealth advisor
research_request = (
    "Research the current state of sustainable investment funds (ESG) "
    "and summarize the findings for a client considering adding ESG exposure "
    "to their portfolio. Include recent performance trends and risk considerations."
)

print(f"📋 Research Request: {research_request}")

## 5️⃣ Run the Compliance-Reviewed Research Workflow

### ⚠️ IMPORTANT: How to Respond to Plan Review

When prompted for plan review, **an input box will appear at the top of VS Code**.

> **You MUST type your response in the input box, then press Enter.**  
> - Press **Enter with empty input** to **approve** the plan
> - Type **feedback text** then Enter to **revise** the plan with your feedback

### Example Responses:

| Input | Action |
|-------|--------|
| *(empty - just press Enter)* | Approve the plan as-is |
| `Add risk disclosures` | Revise plan with your feedback |
| `Include diversification recommendations` | Revise plan with your feedback |

In [ ]:
async def run_compliance_reviewed_research():
    """Run the investment research workflow with Magentic orchestration and compliance plan review."""
    
    print("\n🚀 Starting Magentic research workflow (compliance review required)...")
    print("=" * 60)
    
    pending_request: WorkflowEvent | None = None
    pending_responses: dict[str, MagenticPlanReviewResponse] | None = None
    final_response: AgentResponse | None = None
    review_count = 0

    while not final_response:
        # First iteration uses run with research request
        # Subsequent iterations resume with plan review responses
        if pending_responses is not None:
            stream = workflow.run(stream=True, responses=pending_responses)
        else:
            stream = workflow.run(research_request, stream=True)

        last_message_id: str | None = None
        async for event in stream:
            # Stream intermediate agent outputs
            if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
                message_id = event.data.message_id
                if message_id != last_message_id:
                    if last_message_id is not None:
                        print("\n")
                    print(f"\n- {event.executor_id}:", end=" ", flush=True)
                    last_message_id = message_id
                print(event.data, end="", flush=True)

            # Capture plan review requests from the Magentic manager
            elif event.type == "request_info" and event.request_type is MagenticPlanReviewRequest:
                pending_request = event

            # Capture final synthesized response from the manager
            elif event.type == "output" and isinstance(event.data, AgentResponse):
                final_response = event.data

        pending_responses = None

        # Handle plan review request if any
        if pending_request is not None:
            event_data = cast(MagenticPlanReviewRequest, pending_request.data)
            review_count += 1

            print("\n\n" + "=" * 60)
            print(f"🏦 COMPLIANCE PLAN REVIEW #{review_count}")
            print("=" * 60)

            # Show current progress if this is a replan
            if event_data.current_progress is not None:
                print("\n📊 Current Progress Ledger:")
                print(json.dumps(event_data.current_progress.to_dict(), indent=2))
                print()

            print(f"\n📋 Proposed Plan:\n{event_data.plan.text}\n")
            print("-" * 60)
            print("👨‍💼 COMPLIANCE OFFICER: Review the plan above")
            print("  • Press Enter (empty) to APPROVE the plan")
            print("  • Type feedback to REVISE the plan")
            print("-" * 60)

            reply = await asyncio.get_event_loop().run_in_executor(
                None, 
                input, 
                "Compliance Decision: "
            )

            if reply.strip() == "":
                print("\n✅ Plan APPROVED by compliance.\n")
                pending_responses = {pending_request.request_id: event_data.approve()}
            else:
                print(f"\n📝 Plan REVISED with feedback: {reply}\n")
                pending_responses = {pending_request.request_id: event_data.revise(reply)}
            pending_request = None

    # Display final output - the manager's synthesized answer
    print("\n\n" + "=" * 60)
    print("📊 INVESTMENT RESEARCH REPORT COMPLETE")
    print("=" * 60)
    if final_response and final_response.messages:
        print(final_response.messages[-1].text)

    print(f"\n✅ Completed with {review_count} compliance review(s)")
    print("\n" + "=" * 60)
    print("⚠️ DISCLAIMER: This is for demonstration purposes only.")
    print("   Actual investment research requires full compliance review.")

## 6️⃣ Run the Research Workflow

In [ ]:
await run_compliance_reviewed_research()

## 📝 Key Takeaways

### Compliance Plan Review for FSI

| Benefit | Description |
|---------|-------------|
| **Regulatory Compliance** | Ensure research meets SEC/FINRA guidelines |
| **Quality Assurance** | Verify methodology before execution |
| **Risk Management** | Human approval for client-facing materials |
| **Audit Trail** | Document compliance decisions |

### Industry Use Cases for Plan Review

| Use Case | Review Focus |
|----------|-------------|
| Investment Research | Methodology, disclosures, suitability |
| Client Communications | Regulatory compliance, accuracy |
| Trade Recommendations | Risk assessment, conflict checks |
| Financial Reporting | Data sources, calculation methods |

### Production Considerations

| Consideration | Recommendation |
|---------------|----------------|
| **Audit Trail** | Log all compliance decisions with timestamps |
| **Timeout Handling** | Auto-escalate if review not completed in SLA |
| **Role-Based Access** | Only authorized compliance officers can approve |
| **Multi-Level Approval** | Senior approval for high-value research |